# ViT Sprint 3 — Fine-tune Funnel (Colab A100)

Runs the **five** locked Sprint 3 fold-0 fine-tune candidates from
`007-vit-finetune.md` via `scripts/train.py`. Colab is a GPU runner only — all
model / training / metric logic lives in the repository.

**Dataset on Drive (read this):** copying ~10k loose images over the Drive FUSE
mount is throttled and takes *hours*. Instead, upload the dataset as **one
archive** — `labeled-images.zip` or `labeled-images.tar`, containing the
`labeled-images/` folder at its root — to
`MyDrive/hyperkvasir-multi-cnn-fusion/data/hyperkvasir/`. Cell 5 detects the
archive, copies the single file, and extracts it locally (minutes). It falls
back to per-file copy only if no archive is found.

**Gates (VLD-11):** training is the only A100-eligible stage. The env cell
hard-asserts an A100; `scripts/train.py` also aborts off-A100 internally. Do
**not** pass `--allow-non-a100` for real runs, and do not recompute frozen
feature caches here.

**Overnight-safe:** cell 7 streams live progress, **restores finished runs from
Drive** after a session reset, **skips** runs that already have `metrics.json`,
and **backs each finished run up to Drive immediately** — so if the session dies
while you sleep, you re-run cell 7 and it continues from the unfinished models.

**Candidates (fold 0 only, VLD-12):**
`02_single_swin_t`, `04_pair_vit_b_swin_t_concat`, `05_pair_vit_b_beit_b_concat`,
`09_pair_swin_t_beit_b_weighted`, `11_triple_weighted` (all `_finetune_official`).

Prerequisite: the `sprint3/vit-finetune-funnel` branch must be pushed to GitHub
before running cell 2.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the Sprint 3 branch and record the commit
import os, subprocess
REPO_URL = 'https://github.com/YasinEkici/hyperkvasir-multi-backbone-fusion.git'
BRANCH = 'sprint3/vit-finetune-funnel'
REPO_DIR = '/content/hyperkvasir-multi-backbone-fusion'
# Public repo: no token needed. For a private repo, set
# os.environ['GITHUB_TOKEN'] = '...' in a scratch cell BEFORE running this one.
token = os.environ.get('GITHUB_TOKEN', '')
clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@') if token else REPO_URL
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only'], check=True)
elif os.path.exists(REPO_DIR):
    raise RuntimeError(f'Path exists but is not a git repo: {REPO_DIR}. Restart the runtime.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', clone_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git rev-parse HEAD

In [ ]:
# 3. Build the isolated Colab env and hard-assert an A100 (VLD-11)
import os
REPO_DIR = '/content/hyperkvasir-multi-backbone-fusion'
os.chdir(REPO_DIR)
required_files = [
    'pyproject.toml',
    'env/requirements-colab.txt',
    'configs/vit/training/vit_finetune.yaml',
    'configs/vit/experiment_matrix.yaml',
]
missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    !git branch --show-current
    !git rev-parse HEAD
    raise FileNotFoundError(f'Missing required repo files: {missing}. Push Slices 1-3 to GitHub and rerun cell 2.')
!python -m pip install -q uv
!uv venv --python 3.11 .venv
# Use `uv pip install -r` (resolves transitive deps); never `uv sync` here
# (local pyproject is pinned to CUDA 13.2 for the RTX 5080).
!uv pip install --python .venv/bin/python -r env/requirements-colab.txt
!uv run --no-sync python -c "import torch, timm; print('torch', torch.__version__, 'timm', timm.__version__); assert torch.cuda.is_available(), 'CUDA unavailable'; name=torch.cuda.get_device_name(0); print('device', name); assert 'A100' in name, f'A100 required, found {name}'; print(torch.ones(1, device='cuda'))"

In [ ]:
# 4. CONTROL PANEL — the five locked candidates + Drive root. Edit only here.
import os
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive/hyperkvasir-multi-cnn-fusion'
EXPERIMENTS = [
    '02_single_swin_t_finetune_official',          # smoke first (cheapest)
    '04_pair_vit_b_swin_t_concat_finetune_official',
    '05_pair_vit_b_beit_b_concat_finetune_official',
    '09_pair_swin_t_beit_b_weighted_finetune_official',
    '11_triple_weighted_finetune_official',         # heaviest (triple)
]
print('Drive root:', os.environ['DRIVE_ROOT'])
print('Experiments:', *EXPERIMENTS, sep='\n  ')

In [ ]:
# 5. Stage the dataset to data/raw/hyperkvasir/labeled-images.
#    PREFERS a single archive on Drive (one sequential read — fast, avoids the
#    Drive FUSE small-file throttling that makes 10k per-file copies take hours).
#    Put labeled-images.zip (or .tar) next to the folder on Drive (see README
#    cell). Falls back to per-file copy if no archive exists. Verifies the staged
#    .jpg count against the fold manifest. Fine-tune reads images, not caches.
import os, csv, shutil
from pathlib import Path
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
DRIVE = os.environ['DRIVE_ROOT']
dst = Path('data/raw/hyperkvasir/labeled-images')
with open('data/splits/hyperkvasir_official_5fold/fold_0.csv', newline='', encoding='utf-8-sig') as f:
    expected = sum(1 for _ in csv.DictReader(f))  # 10662 image rows
def count_images(p):
    # Tree also holds image-labels.csv + license.txt — count only .jpg/.jpeg.
    return sum(1 for _, _, fs in os.walk(p) for f in fs
               if f.lower().endswith(('.jpg', '.jpeg'))) if Path(p).exists() else 0
if dst.exists() and count_images(dst) == expected:
    print(f'[skip] already staged: {expected} images')
else:
    if dst.exists():
        print(f'[clean] partial copy {count_images(dst)}/{expected} -> removing', flush=True)
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    archive = next((c for c in [
        f'{DRIVE}/data/hyperkvasir/labeled-images.zip',
        f'{DRIVE}/data/hyperkvasir/labeled-images.tar',
        f'{DRIVE}/data/labeled-images.zip',
    ] if os.path.exists(c)), None)
    if archive:
        local_arc = '/content/' + os.path.basename(archive)
        print(f'[archive] copying {archive} (one big file)...', flush=True)
        shutil.copy(archive, local_arc)
        print('[archive] extracting...', flush=True)
        shutil.unpack_archive(local_arc, 'data/raw/hyperkvasir')  # archive root = labeled-images/
    else:
        src = Path(f'{DRIVE}/data/hyperkvasir/labeled-images')
        if not src.is_dir():
            raise FileNotFoundError(f'No archive and no folder on Drive: {src}')
        files = sorted(p for p in src.rglob('*') if p.is_file())
        print(f'[per-file] no archive on Drive; copying {len(files)} files (SLOW; consider an archive)...', flush=True)
        for i, sp in enumerate(files, 1):
            tp = dst / sp.relative_to(src)
            tp.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(sp, tp)  # data only — fewer FUSE syscalls than copy2
            if i % 1000 == 0 or i == len(files):
                print(f'  {i}/{len(files)}', flush=True)
n = count_images(dst)
assert n == expected, f'staged {n} images != expected {expected} (archive must contain labeled-images/ at its root)'
print(f'[OK] staged {n} images -> {dst}')

In [ ]:
# 6. Dataset + git provenance gate (CNN D-09 gate; VLD-11). Idempotent.
#    NOTE: we hash the LOCAL staged tree (fast) against the manifest + git SHA.
#    We do NOT re-hash the Drive source — reading 10k files over FUSE is
#    prohibitively slow. Integrity is covered upstream: the archive was verified
#    to hold 10662 jpg at build time and cell 5 re-checks the staged count. The
#    Drive archive is recorded as approved_source in the provenance record.
import os, shutil, subprocess
from pathlib import Path
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
prov = Path('results/vit/runs/sprint3_vit_finetune')
if prov.exists():
    shutil.rmtree(prov)
os.environ['EXPECTED_GIT_SHA'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
!uv run --no-sync python scripts/check_provenance.py --run-id sprint3_vit_finetune --source-dataset-root data/raw/hyperkvasir/labeled-images --staged-dataset-root data/raw/hyperkvasir/labeled-images --manifest data/splits/hyperkvasir_official_5fold/fold_0.csv --approved-source "$DRIVE_ROOT/data/hyperkvasir/labeled-images.tar" --expected-git-sha $EXPECTED_GIT_SHA --device cuda --output-root results/vit/runs

In [ ]:
# 7. Train the five fine-tune rows on fold 0 — overnight-safe & resumable.
#    Restores finished runs from Drive, skips completed ones, streams live
#    progress (python -u), and backs each finished run up to Drive immediately.
import os, shutil, subprocess
from pathlib import Path
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
runs = Path('results/vit/runs')
backup = Path(os.environ['DRIVE_ROOT']) / 'returned_outputs' / 'sprint3_vit_finetune'
backup.mkdir(parents=True, exist_ok=True)

# Restore previously-finished runs from Drive (survives a fresh session).
for exp in EXPERIMENTS:
    if (backup / exp / 'metrics.json').exists() and not (runs / exp / 'metrics.json').exists():
        shutil.copytree(backup / exp, runs / exp, dirs_exist_ok=True)
        print(f'[restored from Drive] {exp}')

done = [e for e in EXPERIMENTS if (runs / e / 'metrics.json').exists()]
todo = [e for e in EXPERIMENTS if e not in done]
print(f'done ({len(done)}): {done}')
print(f'todo ({len(todo)}): {todo}\n', flush=True)

env = {**os.environ, 'PYTHONUNBUFFERED': '1'}  # stream child stdout live
for exp in todo:
    print(f'\n===== [run] {exp} =====', flush=True)
    subprocess.run(
        ['uv', 'run', '--no-sync', 'python', '-u', 'scripts/train.py',
         '--config', 'configs/vit/experiment_matrix.yaml',
         '--experiment', exp, '--device', 'cuda'],
        check=True, env=env,
    )
    shutil.copytree(runs / exp, backup / exp, dirs_exist_ok=True)  # back up now
    print(f'[backed up to Drive] {exp}', flush=True)
print('\n[done] all requested fine-tune runs finished')

In [ ]:
# 8. Artifact + finite-metric checklist (hard-fail on any problem)
import json, math
from pathlib import Path
runs = Path('results/vit/runs')
required = ['metrics.json', 'config.yaml', 'predictions.npz', 'best.pt']
problems = []
for exp in EXPERIMENTS:
    present = {r: (runs / exp / r).exists() for r in required}
    if all(present.values()):
        m = json.load((runs / exp / 'metrics.json').open())['test']
        finite = all(isinstance(m.get(k), (int, float)) and math.isfinite(m[k])
                     for k in ('macro_f1', 'accuracy', 'macro_precision', 'macro_recall'))
        print(f"{exp}: f1={m['macro_f1']:.4f} acc={m['accuracy']:.4f} finite={finite}")
        if not finite:
            problems.append(f'{exp}: non-finite test metric')
    else:
        print(f'{exp}: MISSING {present}')
        problems.append(f'{exp}: missing artifacts {present}')
if problems:
    raise RuntimeError('Artifact/metric problems:\n  ' + '\n  '.join(problems))
print('\n[OK] all five runs have the 4 artifacts and finite test metrics')

In [ ]:
# 9. Zip the five run dirs for direct download (also backed up per-model in 7).
import zipfile
from pathlib import Path
runs = Path('results/vit/runs')
zip_path = '/content/vit_finetune_runs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for exp in EXPERIMENTS:
        for fp in (runs / exp).rglob('*'):
            if fp.is_file():
                z.write(fp, fp.relative_to(runs))
print('[OK] zip:', zip_path)

In [ ]:
# 10. Download the zip to your machine (unzip into local results/vit/runs/).
from google.colab import files
files.download('/content/vit_finetune_runs.zip')